# 03 · Milestone 3 evaluation
Reads the outputs of `scripts/eval_m3.py` (run on the HPC GPU) and builds every table and plot for the report.

* **Models:** 2 embedding-injection methods (prefix conditioning, cross-attention) × 3 GPT-2 fine-tuning strategies (full, LoRA, frozen = prefix tuning).
* **Test set:** the first 1,000 COCO val2017 images, with 5 human captions each. These images were never used for checkpoint selection, which used the other 4,000 val images.
* **Metrics:** BLEU-1..4, METEOR, ROUGE-L, CIDEr (pycocoevalcap); CLIPScore for image–caption alignment; distinct-n for diversity.

In [ ]:
import os, json, glob, textwrap
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from PIL import Image
pd.set_option('display.precision', 3); pd.set_option('display.width', 200)
RES = '../results/m3'; FIG = f'{RES}/figures'; TAB = f'{RES}/tables'
DATA = os.environ.get('GP1_DATA', '/scratch/hu.zhim/gp1/data')
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
INJ = {'prefix': 'Prefix conditioning', 'cross': 'Cross-attention'}
FT = {'full': 'Full fine-tuning', 'lora': 'LoRA', 'frozen': 'Prefix tuning (frozen GPT-2)'}
COL = {'prefix': '#257de0', 'cross': '#f47230'}; MARK = {'prefix': 'o', 'cross': 's'}
INK, INK2, GRID = '#191919', '#666666', '#e6e6e6'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.edgecolor': '#cfcfcf', 'axes.labelcolor': INK2,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True, 'figure.dpi': 110})
M = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f'{RES}/*/metrics.csv'))], ignore_index=True)
CFG = {os.path.basename(os.path.dirname(f)): json.load(open(f)) for f in glob.glob(f'{RES}/*/config.json')}
METRICS = ['BLEU-1', 'BLEU-4', 'METEOR', 'ROUGE-L', 'CIDEr', 'CLIPScore']
print(len(M), 'evaluation rows |', sorted(M.tag.unique()))

## 1. Training summary
Best epoch = the epoch with the lowest validation loss on the 4,000 selection images; that checkpoint is evaluated.

In [ ]:
rows = []
for tag, c in sorted(CFG.items()):
    ep = pd.read_csv(f'{RES}/{tag}/epoch_log.csv')
    rows.append({'Injection': INJ[c['injection']], 'Fine-tuning': FT[c['finetune']], 'tag': tag,
                 'Trainable params (M)': c['params_trainable_M'], 'Best epoch': c['best_epoch'],
                 'Best val loss': c['best_val_loss'], 'Val loss @ epoch 5': ep.val_loss.iloc[-1], 'Train min': c['train_minutes']})
train_tab = pd.DataFrame(rows); train_tab.to_csv(f'{TAB}/training_summary.csv', index=False); train_tab

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, inj in zip(axes, INJ):
    for ft, ls in zip(FT, ['-', '--', ':']):
        ep = pd.read_csv(f'{RES}/{inj}_{ft}/epoch_log.csv').query('epoch > 0')
        ax.plot(ep.epoch, ep.val_loss, ls, color=COL[inj], lw=2, marker=MARK[inj], ms=6, mec='white', mew=1.5)
        ax.annotate(FT[ft].split(' (')[0], (ep.epoch.iloc[-1], ep.val_loss.iloc[-1]), xytext=(6, 0), textcoords='offset points', va='center', fontsize=8.5, color=INK)
    ax.set(title=INJ[inj], xlabel='Epoch', xlim=(0.7, 6.6), xticks=range(1, 6)); ax.title.set_color(INK)
axes[0].set_ylabel('Validation loss (per token)')
plt.tight_layout(); plt.savefig(f'{FIG}/val_loss_by_strategy.png', dpi=200); plt.show()

## 2. Fine-tuning strategies × injection methods (beam search, width 3)

In [ ]:
main = M[M.strategy == 'beam:3'].copy()
main['Injection'] = main.injection.map(INJ); main['Fine-tuning'] = main.finetune.map(FT)
main = main.merge(train_tab[['tag', 'Trainable params (M)']], on='tag')
main_tab = main[['Injection', 'Fine-tuning', 'Trainable params (M)'] + METRICS].sort_values(['Injection', 'Fine-tuning'])
main_tab.to_csv(f'{TAB}/finetuning_comparison_beam3.csv', index=False); main_tab

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
x = np.arange(3); w = 0.36
for ax, met in zip(axes, ['CIDEr', 'METEOR', 'CLIPScore']):
    for k, inj in enumerate(INJ):
        v = [main.query('injection == @inj and finetune == @ft')[met].item() for ft in FT]
        bars = ax.bar(x + (k - 0.5) * w, v, w - 0.03, color=COL[inj], label=INJ[inj])
        for b, val in zip(bars, v):
            ax.text(b.get_x() + b.get_width() / 2, val, f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, color=INK)
    ax.set_xticks(x, ['Full', 'LoRA', 'Prefix tuning\n(frozen)']); ax.set_title(met, color=INK); ax.grid(axis='x', visible=False)
    lo = min(main[met]); ax.set_ylim(lo * 0.85 if met == 'CLIPScore' else 0, max(main[met]) * 1.12)
axes[0].legend(frameon=False, fontsize=8.5, loc='upper right')
plt.tight_layout(); plt.savefig(f'{FIG}/finetuning_comparison.png', dpi=200); plt.show()

## 3. Parameter sensitivity: beam width and temperature (full fine-tuning)

In [ ]:
full = M[M.finetune == 'full'].copy()
beam = full[full.family.isin(['greedy', 'beam width'])].copy(); beam['width'] = beam.value.fillna(1).astype(int)
temp = full[full.family == 'temperature'].copy()
beam_tab = beam.pivot_table(index='width', columns='injection', values=['CIDEr', 'METEOR', 'BLEU-4', 'distinct-2', 'sec_per_image'])
temp_tab = temp.pivot_table(index='value', columns='injection', values=['CIDEr', 'METEOR', 'BLEU-4', 'CLIPScore', 'distinct-2', 'unique_captions'])
beam_tab.to_csv(f'{TAB}/sensitivity_beam_width.csv'); temp_tab.to_csv(f'{TAB}/sensitivity_temperature.csv')
display(beam_tab.round(3)); display(temp_tab.round(3))

In [ ]:
def sweep(df, xcol, mets, fname, xlabel, logx=False):
    fig, axes = plt.subplots(1, len(mets), figsize=(3.4 * len(mets), 3.3))
    for ax, met in zip(axes, mets):
        for inj in INJ:
            d = df[df.injection == inj].sort_values(xcol)
            ax.plot(d[xcol], d[met], color=COL[inj], lw=2, marker=MARK[inj], ms=6, mec='white', mew=1.5, label=INJ[inj])
            ax.annotate(INJ[inj].split()[0], (d[xcol].iloc[-1], d[met].iloc[-1]), xytext=(4, 0), textcoords='offset points', va='center', fontsize=8, color=INK)
        ax.set(xlabel=xlabel, title=met); ax.title.set_color(INK)
        if logx: ax.set_xscale('log', base=2); ax.set_xticks(sorted(df[xcol].unique()), [str(int(v)) for v in sorted(df[xcol].unique())])
    axes[0].legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.savefig(f'{FIG}/{fname}', dpi=200); plt.show()

sweep(beam, 'width', ['CIDEr', 'METEOR', 'distinct-2', 'sec_per_image'], 'sensitivity_beam_width.png', 'Beam width (1 = greedy)', logx=True)
sweep(temp, 'value', ['CIDEr', 'METEOR', 'CLIPScore', 'distinct-2'], 'sensitivity_temperature.png', 'Sampling temperature')

## 4. Decoding strategies: accuracy vs. diversity
Every decoding setting evaluated with the two fully fine-tuned models.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for inj in INJ:
    d = full[full.injection == inj]
    ax.scatter(d['distinct-2'], d['CIDEr'], s=60, color=COL[inj], marker=MARK[inj], edgecolor='white', linewidth=1.5, label=INJ[inj], zorder=3)
    for _, r in d.iterrows():
        if inj == 'cross' and r.family in ('greedy', 'beam width', 'temperature', 'top-p'):
            lab = ('greedy' if r.family == 'greedy' else f'beam {int(r.value)}' if r.family == 'beam width'
                   else f'T={r.value:g}' if r.family == 'temperature' else f'top-p {r.value:g}')
            ax.annotate(lab, (r['distinct-2'], r.CIDEr), xytext=(5, 3), textcoords='offset points', fontsize=7.5, color=INK2)
ax.set(xlabel='Diversity: distinct-2 (unique bigrams / all bigrams)', ylabel='Accuracy: CIDEr')
ax.legend(frameon=False, fontsize=8.5)
plt.tight_layout(); plt.savefig(f'{FIG}/accuracy_vs_diversity.png', dpi=200); plt.show()
dec_tab = full[['injection', 'strategy'] + METRICS + ['distinct-2', 'unique_captions', 'avg_words', 'sec_per_image']]
dec_tab.to_csv(f'{TAB}/decoding_strategies.csv', index=False)

## 5. Qualitative alignment between image and caption
Best model and setting (by CIDEr). The examples are 8 random test images plus the 3 highest- and 3 lowest-CLIPScore captions. We also check whether the reference-free CLIPScore agrees with the reference-based CIDEr for each image.

In [ ]:
best = main.sort_values('CIDEr').iloc[-1]; print('best model:', best.tag, '| beam:3 | CIDEr %.3f' % best.CIDEr)
caps = pd.DataFrame([json.loads(l) for l in open(f'{RES}/{best.tag}/captions.jsonl')])
piv = caps.pivot_table(index='file_name', columns='strategy', values='caption', aggfunc='first')
sc = caps[caps.strategy == 'beam:3'].set_index('file_name')
refs = {json.loads(l)['file_name']: json.loads(l)['captions'][0] for l in open(f'{DATA}/val/captions.jsonl')}
r = np.corrcoef(sc.cider, sc.clipscore)[0, 1]
print(f'per-image correlation CIDEr vs CLIPScore (beam:3): r = {r:.2f}')
rnd = list(pd.Series(sc.index).sample(8, random_state=1))
top, bot = list(sc.clipscore.nlargest(3).index), list(sc.clipscore.nsmallest(3).index)
QI = f'{RES}/qual_images'; os.makedirs(QI, exist_ok=True)
qual = []
for group, names in [('random', rnd), ('highest CLIPScore', top), ('lowest CLIPScore', bot)]:
    for n in names:
        Image.open(f'{DATA}/val/{n}').convert('RGB').resize((300, 300)).save(f'{QI}/{os.path.basename(n)}', quality=90)
        qual.append({'group': group, 'file_name': n, 'reference': refs[n], 'greedy': piv.loc[n, 'greedy'], 'beam:3': piv.loc[n, 'beam:3'],
                     'nucleus:0.9:1.0': piv.loc[n, 'nucleus:0.9:1.0'], 'CIDEr': sc.loc[n, 'cider'], 'CLIPScore': sc.loc[n, 'clipscore']})
qual = pd.DataFrame(qual); qual.to_csv(f'{TAB}/qualitative_examples.csv', index=False)
json.dump({'best_tag': best.tag, 'corr_cider_clipscore': round(float(r), 3)}, open(f'{TAB}/qualitative_summary.json', 'w'))

fig, axes = plt.subplots(2, 4, figsize=(15, 10.5))
for ax, (_, q) in zip(axes.flat, qual[qual.group == 'random'].iterrows()):
    ax.imshow(Image.open(f"{QI}/{os.path.basename(q.file_name)}")); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    txt = [f"{k}: {q[k]}" for k in ['greedy', 'beam:3', 'nucleus:0.9:1.0', 'reference']] + [f"CLIPScore {q.CLIPScore:.2f} | CIDEr {q.CIDEr:.2f}"]
    ax.set_xlabel('\n'.join(textwrap.fill(t, 44) for t in txt), fontsize=8.3, loc='left', color=INK)
plt.tight_layout(); plt.savefig(f'{FIG}/qualitative_grid.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(sc.clipscore, sc.cider, s=10, color=COL[best.injection], alpha=0.5, edgecolor='none')
ax.set(xlabel='CLIPScore (reference-free)', ylabel='CIDEr (vs. 5 references)', title=f'Per-image agreement, r = {r:.2f}'); ax.title.set_color(INK)
plt.tight_layout(); plt.savefig(f'{FIG}/cider_vs_clipscore.png', dpi=200); plt.show()
qual[qual.group != 'random'][['group', 'beam:3', 'reference', 'CLIPScore', 'CIDEr']]